# 第 3 周练习：分词器（Tokenizer）+ 提示预算分析器

## 练习目标

比较多个 Hugging Face **分词器**对同一段提示（prompt）的 **token 计数**，并估算它吃掉了模型 **上下文窗口（context window）** 的百分之多少。

另外提供一个简单的 **提示修剪助手**：把文本裁到指定 token 预算内。

## 和本课 Week 3 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Tokenizer / 子词切分 | `AutoTokenizer` + `encode` / `decode` |
| 上下文长度预算 | `tokens / context` 百分比报告 |
| 超长提示处理 | `trim_to_budget`：截断 token 再解码 |

## 怎么跑

1. 若本机还没有依赖，可取消注释安装格后运行
2. 从上到下依次运行；可把 `PROMPT` 换成你自己的英文提示再对比
3. 最后一格会把各模型提示裁到约 80 tokens 并打印结果


In [ ]:
# ========== 可选安装：本机缺包时再取消注释 ==========
# transformers：提供 AutoTokenizer
# sentencepiece：部分模型（如 T5）分词后端需要
# !pip -q install transformers sentencepiece




In [ ]:
# ========== 导入：正则 + Hugging Face 分词器 ==========

# re：标准库正则（本练习主流程未必用到，但保留原导入）
import re
# AutoTokenizer：按 model id 自动加载对应分词器
from transformers import AutoTokenizer




In [ ]:
# ========== 待比较的公开模型：name=Hub id，context≈常见窗口 ==========
# context 是近似默认值，用于预算百分比，不是从 Hub 动态读取的硬限制

MODELS = [
    {'name': 'gpt2', 'context': 1024},
    {'name': 'distilbert-base-uncased', 'context': 512},
    {'name': 'bert-base-uncased', 'context': 512},
    {'name': 'google/flan-t5-small', 'context': 512},
]




In [ ]:
# ========== 示例提示：换成你自己的英文 prompt 再跑预算 ==========
# 三引号内是发给模型的文本，影响 tokenize 结果，禁止改译

PROMPT = '''
You are a helpful assistant.
Summarize the following text and list 3 action items.

Meeting transcript:
We discussed the Q2 launch plan, timelines, and dependencies.
Engineering will finalize the API integration by next Friday.
Marketing will prepare the announcement draft by Monday.
Support needs a short FAQ for common issues and escalation steps.
Risks include vendor delays and limited QA bandwidth.

Please write a concise summary and three action items.
'''




In [ ]:
# ========== 分词器缓存 + 计数 + 预算报告 ==========

# 全局缓存：同一 model_name 只 from_pretrained 一次，避免重复下载
_TOKENIZERS = {}

def get_tokenizer(model_name: str):
    # 缓存未命中时再加载；use_fast=True 优先用 Rust 快速分词器
    if model_name not in _TOKENIZERS:
        _TOKENIZERS[model_name] = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    return _TOKENIZERS[model_name]

def count_tokens(model_name: str, text: str) -> int:
    # 取分词器，把文本编成 token id 列表
    tok = get_tokenizer(model_name)
    # add_special_tokens=False：不计 BOS/EOS/CLS 等特殊符，只量「正文」长度
    return len(tok.encode(text, add_special_tokens=False))

def budget_report(text: str):
    # 逐模型统计：token 数、上下文、占用百分比
    rows = []
    for m in MODELS:
        # 当前模型对该 text 的 token 数
        n = count_tokens(m['name'], text)
        # 近似上下文窗口上限
        ctx = m['context']
        # 占用百分比，保留两位小数
        pct = round((n / ctx) * 100, 2)
        # 一行报告字典（键名保持英文，便于打印/后续处理）
        rows.append({
            'model': m['name'],
            'tokens': n,
            'context': ctx,
            'pct_of_context': pct
        })
    return rows




In [ ]:
# ========== 打印预算报告：看 PROMPT 在各模型下吃掉多少窗口 ==========

# 对示例 PROMPT 生成各模型的 token / 占比行
report = budget_report(PROMPT)
# 逐行打印 dict，方便肉眼对比
for row in report:
    print(row)




In [ ]:
# ========== 提示修剪：按 token 预算截断后再 decode 回文本 ==========

def trim_to_budget(model_name: str, text: str, max_tokens: int) -> str:
    # 取对应模型的分词器
    tok = get_tokenizer(model_name)
    # 先 encode 成 token id 序列（不计特殊符）
    tokens = tok.encode(text, add_special_tokens=False)
    # 已在预算内：原样返回，不改动文本
    if len(tokens) <= max_tokens:
        return text
    # 超长：只保留前 max_tokens 个 id（硬截断）
    trimmed_tokens = tokens[:max_tokens]
    # decode 回字符串；skip_special_tokens 去掉特殊符
    trimmed_text = tok.decode(trimmed_tokens, skip_special_tokens=True)
    # 去掉尾部空白后加省略号标记（字符串字面量保持原样，含换行）
    return trimmed_text.rstrip() + '...
'

# 示例：每个模型把 PROMPT 裁到最多 80 个 token，再打印预览
for m in MODELS:
    trimmed = trim_to_budget(m['name'], PROMPT, max_tokens=80)
    print('
---', m['name'], '---')
    print(trimmed)

